# Cross-validation depth 2: Odra vs simulator

Evaluates **Odra** and **Simulator** ansätze from [`setup/cross_validation/Models/Training/depth 2/`](setup/cross_validation/Models/Training/depth%202/) (**trimmed reverse-q0** circuits — same definitions as those notebooks), for **every fold** (k = 1 … 5).

- **Test CSV (per fold):** [`setup/cross_validation/Data/fold_k/test_data.csv`](setup/cross_validation/Data/fold_1/test_data.csv)
- **Weights:** [`setup/cross_validation/Models/Weights/depth 2/`](setup/cross_validation/Models/Weights/depth%202/)
  - Odra (clean): `Odra/fold_k/Odra_fold_{k}_depth_2_epoch_30_weights.pth`
  - Simulator (clean / **ideal**): `Simulator/fold_k/Simulator_fold_{k}_depth_2_epoch_30_ideal_weights.pth` (not the non-`ideal` `epoch_30` file in the same folder)

- **StatevectorEstimator:** deterministic baseline (std reported as 0).
- **IQM** (optional): set **`RUN_IQM_HARDWARE = True`** for repeated shot evaluation on **IQM Spark ODRA**; otherwise only statevector metrics are computed and IQM columns are **NaN**.

There is **no** `odra_native` in this CV tree.

**Dependencies:** Qiskit 2.x, `qiskit-machine-learning`, `iqm-client[qiskit]` (if using IQM), PyTorch, scikit-learn, pandas. Prefer the project **uv** environment ([`pyproject.toml`](../../pyproject.toml)).



In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# --- Evaluation configuration ---
NUM_QUBITS = 5
ANSATZ_DEPTH = 2
RANDOM_SEED = 42
K_FOLDS = 5

# Shots per circuit on IQM (only if RUN_IQM_HARDWARE)
EVAL_SHOTS = 2048

# At least 10 repeats per (fold, ansatz) on hardware when RUN_IQM_HARDWARE is True
N_REPEATS = 10

# Set True to prompt for IQM token and run shot-based evaluation on hardware
RUN_IQM_HARDWARE = False

# Cross-validation layout (relative to project root)
CROSS_VAL_DIR = "tests/ansatz_odra_evaluation/setup/cross_validation"

# CSV output: tests/ansatz_odra_evaluation/outputs/run_<timestamp>/



In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.primitives import PrimitiveResult, PubResult, StatevectorEstimator
from qiskit.primitives.base import BaseEstimatorV2
from qiskit.primitives.containers.data_bin import DataBin
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.gradients import ParamShiftEstimatorGradient
from qiskit_machine_learning.neural_networks import EstimatorQNN
from sklearn.metrics import accuracy_score, f1_score

try:
    from tqdm import tqdm

    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False


def _project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "cross_validation").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root (folder with cross_validation/).")


def load_fold_test_data(root: Path, cross_val_relative: str, fold: int) -> tuple[np.ndarray, np.ndarray]:
    path = root / cross_val_relative / "Data" / f"fold_{fold}" / "test_data.csv"
    if not path.is_file():
        raise FileNotFoundError(path)
    data = np.loadtxt(path, delimiter=",", skiprows=1, ndmin=2)
    if data.shape[1] != 6:
        raise ValueError(f"Expected 6 columns in {path}, got {data.shape[1]}")
    x = data[:, :5].astype(np.float32)
    y = data[:, 5].astype(np.float32)
    if set(np.unique(y)).issubset({0.0, 1.0}):
        y = 2.0 * y - 1.0
    return x, y


def predictions_to_labels(predictions: np.ndarray) -> np.ndarray:
    predictions = np.asarray(predictions).reshape(-1)
    return np.where(predictions > 0, 1, -1).astype(np.float32)


ROOT = _project_root()
CV_ABS = ROOT / CROSS_VAL_DIR
WEIGHTS_ROOT = CV_ABS / "Models" / "Weights" / f"depth {ANSATZ_DEPTH}"


def fold_test_csv_path(fold: int) -> Path:
    return CV_ABS / "Data" / f"fold_{fold}" / "test_data.csv"


def odra_weights_path(fold: int) -> Path:
    return (
        WEIGHTS_ROOT
        / "Odra"
        / f"fold_{fold}"
        / f"Odra_fold_{fold}_depth_{ANSATZ_DEPTH}_epoch_20_weights.pth"
    )


def simulator_ideal_weights_path(fold: int) -> Path:
    return (
        WEIGHTS_ROOT
        / "Simulator"
        / f"fold_{fold}"
        / f"Simulator_fold_{fold}_depth_{ANSATZ_DEPTH}_epoch_20_ideal_weights.pth"
    )


print(f"Project root: {ROOT}")
print(f"Cross-validation dir: {CV_ABS} (exists={CV_ABS.is_dir()})")
print(f"Weights root: {WEIGHTS_ROOT} (exists={WEIGHTS_ROOT.is_dir()})")
print(f"RUN_IQM_HARDWARE={RUN_IQM_HARDWARE}")



In [ ]:
def ansatz_trimmed_reverse_q0_param_count(n_qubits: int, depth: int) -> int:
    """Weights when only the last macro-layer uses the q0-incident reverse trim."""
    m = depth // 2
    if m == 0:
        return 0
    full = 4 * n_qubits
    last = 3 * n_qubits + 2
    return (m - 1) * full + last


def odra_ansatz(n_qubits: int, depth: int) -> QuantumCircuit:
    # Full reverse (4n) on all macro-layers except the last; there trim reverse to q0 edges only (3n+2).
    n_macro = depth // 2
    theta = ParameterVector("theta", ansatz_trimmed_reverse_q0_param_count(n_qubits, depth))
    qc = QuantumCircuit(n_qubits)
    p = 0

    for j in range(n_macro):
        last_layer = j == n_macro - 1

        for i in range(n_qubits):
            qc.ry(theta[p + i], i)
        p += n_qubits

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.rz(theta[p + i], target)
            qc.cz(control, target)
        p += n_qubits

        for i in range(n_qubits):
            qc.rx(theta[p + i], i)
        p += n_qubits

        if last_layer:
            for k in range(2):
                i = k
                control = i
                target = (i - 1) % n_qubits
                qc.ry(theta[p + k], target)
                qc.cz(control, target)
            p += 2
        else:
            for i in range(n_qubits):
                control = i
                target = (i - 1) % n_qubits
                qc.ry(theta[p + i], target)
                qc.cz(control, target)
            p += n_qubits

    assert p == len(theta)
    return qc


def simulator_ansatz(n_qubits: int, depth: int) -> QuantumCircuit:
    n_macro = depth // 2
    theta = ParameterVector("theta", ansatz_trimmed_reverse_q0_param_count(n_qubits, depth))
    qc = QuantumCircuit(n_qubits)
    param_idx = 0

    for j in range(n_macro):
        last_layer = j == n_macro - 1

        for i in range(n_qubits):
            qc.ry(theta[param_idx], i)
            param_idx += 1

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.crx(theta[param_idx], control, target)
            param_idx += 1

        for i in range(n_qubits):
            qc.rx(theta[param_idx], i)
            param_idx += 1

        if last_layer:
            for k in range(2):
                i = k
                control = i
                target = (i - 1) % n_qubits
                qc.cry(theta[param_idx], control, target)
                param_idx += 1
        else:
            for i in range(n_qubits):
                control = i
                target = (i - 1) % n_qubits
                qc.cry(theta[param_idx], control, target)
                param_idx += 1

    assert param_idx == len(theta)
    return qc


_nexp = ansatz_trimmed_reverse_q0_param_count(NUM_QUBITS, ANSATZ_DEPTH)
_od = odra_ansatz(NUM_QUBITS, ANSATZ_DEPTH)
_sim = simulator_ansatz(NUM_QUBITS, ANSATZ_DEPTH)
assert len(list(_od.parameters)) == _nexp
assert len(list(_sim.parameters)) == _nexp
print(f"Ansatz params (trimmed CV depth {ANSATZ_DEPTH}): odra={_nexp}, simulator={_nexp}")



In [ ]:
class SimpleIQMJob:
    def __init__(self, result):
        self._result = result

    def result(self):
        return self._result


class IQMBackendEstimator(BaseEstimatorV2):
    def __init__(self, backend, options=None):
        super().__init__()
        self._backend = backend
        self._options = options or {"shots": 100}
        self.timestamp_history = []
        self.total_qpu_time = 0.0

    def _extract_timestamps(self, result):
        try:
            timeline = result._metadata.get("timeline", [])
            if not timeline:
                return None
            timestamps = {}
            for entry in timeline:
                timestamps[entry.status] = entry.timestamp
            return timestamps
        except Exception:
            return None

    def _counts_to_expectation(self, counts):
        if isinstance(counts, list):
            counts = counts[0]
        shots = sum(counts.values())
        count_0 = sum(c for bitstring, c in counts.items() if bitstring[-1] == "0")
        p0 = count_0 / shots if shots else 0.0
        return p0 - (1 - p0)

    def run(self, pubs, precision=None):
        if not isinstance(pubs, list):
            pubs = [pubs]

        job_results = []
        shots_opt = self._options["shots"]
        max_circuits = self._options.get("max_circuits_per_job")

        base_circuit = pubs[0][0]
        circuit_with_meas = base_circuit.copy()
        if circuit_with_meas.num_clbits == 0:
            circuit_with_meas.measure_all()
        transpiled_qc = transpile(circuit_with_meas, self._backend, optimization_level=0)

        for pub in pubs:
            _, observables, parameter_values = pub
            if parameter_values.ndim == 1:
                parameter_values = [parameter_values]

            bound_circuits = [transpiled_qc.assign_parameters(params) for params in parameter_values]
            n_circuits = len(bound_circuits)
            pub_expectations = []

            for start in range(0, n_circuits, max_circuits or n_circuits):
                end = min(start + (max_circuits or n_circuits), n_circuits)
                batch = bound_circuits[start:end]
                try:
                    job = self._backend.run(batch, shots=shots_opt)
                    result = job.result()

                    ts = self._extract_timestamps(result)
                    if ts:
                        exec_start = ts.get("execution_started")
                        exec_end = ts.get("execution_ended")
                        comp_start = ts.get("compilation_started")
                        comp_end = ts.get("compilation_ended")
                        job_created = ts.get("created")
                        job_completed = ts.get("completed")
                        if exec_start and exec_end:
                            execution_time = (exec_end - exec_start).total_seconds()
                            compile_time = (
                                (comp_end - comp_start).total_seconds() if comp_start and comp_end else 0.0
                            )
                            job_time = (
                                (job_completed - job_created).total_seconds()
                                if job_created and job_completed
                                else 0.0
                            )
                            self.timestamp_history.append(
                                {
                                    "execution_time_qpu": execution_time,
                                    "job_time_total": job_time,
                                    "compile_time": compile_time,
                                    "raw_timestamps": ts,
                                    "n_circuits": len(batch),
                                }
                            )
                            self.total_qpu_time += execution_time

                    counts_list = result.get_counts()
                    if not isinstance(counts_list, list):
                        counts_list = [counts_list]
                    for counts in counts_list:
                        pub_expectations.append(self._counts_to_expectation(counts))
                except Exception as exc:
                    print(f"Batch job failed: {exc}")
                    pub_expectations.extend([0.0] * len(batch))

            data = DataBin(evs=np.array(pub_expectations), shape=(len(pub_expectations),))
            job_results.append(PubResult(data=data))

        return SimpleIQMJob(PrimitiveResult(job_results))

In [ ]:
class HybridModel(nn.Module):
    def __init__(self, ansatz_circuit: QuantumCircuit, num_qubits: int):
        super().__init__()
        self.feature_map = self.angle_encoding(num_qubits)
        self.qc = QuantumCircuit(num_qubits)
        self.qc.compose(self.feature_map, qubits=range(num_qubits), inplace=True)
        self.qc.compose(ansatz_circuit, inplace=True)

        input_params = list(self.feature_map.parameters)
        weight_params = list(ansatz_circuit.parameters)
        observable = SparsePauliOp.from_list([("I" * (num_qubits - 1) + "Z", 1)])
        estimator = StatevectorEstimator(seed=RANDOM_SEED)
        gradient = ParamShiftEstimatorGradient(estimator=estimator)

        self.qnn = EstimatorQNN(
            circuit=self.qc,
            observables=observable,
            input_params=input_params,
            weight_params=weight_params,
            estimator=estimator,
            gradient=gradient,
        )
        self.quantum_layer = TorchConnector(self.qnn)

    def angle_encoding(self, num_qubits: int) -> QuantumCircuit:
        qc_data = QuantumCircuit(num_qubits)
        input_params = ParameterVector("x", num_qubits)
        for i in range(num_qubits):
            qc_data.ry(input_params[i], i)
        return qc_data

    def forward(self, x):
        return self.quantum_layer(x)


def build_statevector_model(ansatz_fn, num_qubits: int, depth: int) -> HybridModel:
    circ = ansatz_fn(num_qubits, depth)
    return HybridModel(circ, num_qubits)


def build_iqm_model(iqm_backend, ansatz_fn, n_shots: int, num_qubits: int, depth: int):
    hw_estimator = IQMBackendEstimator(iqm_backend, options={"shots": n_shots})
    hw_ansatz = ansatz_fn(num_qubits, depth)
    hw_feature_map = HybridModel(hw_ansatz, num_qubits).angle_encoding(num_qubits)

    hw_qc = QuantumCircuit(num_qubits)
    hw_qc.compose(hw_feature_map, qubits=range(num_qubits), inplace=True)
    hw_qc.compose(hw_ansatz, inplace=True)

    observable = SparsePauliOp.from_list([("I" * (num_qubits - 1) + "Z", 1)])
    hw_qnn = EstimatorQNN(
        circuit=hw_qc,
        observables=observable,
        input_params=list(hw_feature_map.parameters),
        weight_params=list(hw_ansatz.parameters),
        estimator=hw_estimator,
    )
    hw_model = TorchConnector(hw_qnn)
    return hw_model, hw_estimator


def _torch_load(path: Path):
    import importlib
    import pickle

    # Always use a fresh binding to the real package (global ``torch`` can be shadowed). Jupyter
    # autoreload sometimes leaves the package without ``._utils`` in ``__dict__``, which breaks
    # ``torch.load`` even with ``weights_only=False``.
    import torch as _tp

    if _tp.__dict__.get("_utils") is None:
        _tp.__dict__["_utils"] = importlib.import_module("torch._utils")

    try:
        return _tp.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return _tp.load(path, map_location="cpu")
    except (AttributeError, pickle.UnpicklingError):
        return _tp.load(path, map_location="cpu", weights_only=False)


def _unwrap_state_dict(obj):
    if isinstance(obj, dict) and "state_dict" in obj and isinstance(obj["state_dict"], dict):
        return obj["state_dict"]
    return obj


def _strip_quantum_layer_prefix(state: dict) -> dict:
    out = {}
    for key, value in state.items():
        if key.startswith("quantum_layer."):
            out[key.replace("quantum_layer.", "", 1)] = value
        else:
            out[key] = value
    return out


def load_checkpoint_hybrid(model: HybridModel, path: Path) -> None:
    raw = _unwrap_state_dict(_torch_load(path))
    if not isinstance(raw, dict):
        raise TypeError(f"Expected dict checkpoint at {path}")
    try:
        model.load_state_dict(raw, strict=True)
        return
    except Exception:
        pass
    stripped = _strip_quantum_layer_prefix(raw)
    model.quantum_layer.load_state_dict(stripped, strict=True)


def load_checkpoint_connector(connector: TorchConnector, path: Path) -> None:
    raw = _unwrap_state_dict(_torch_load(path))
    if not isinstance(raw, dict):
        raise TypeError(f"Expected dict checkpoint at {path}")
    stripped = _strip_quantum_layer_prefix(raw)
    connector.load_state_dict(stripped, strict=True)


def connect_to_iqm_backend():
    base_url = "https://odra5.e-science.pl/"
    token = input("Enter IQM Token: ").strip()
    if not token:
        raise ValueError("An IQM token is required.")
    from iqm.qiskit_iqm import IQMProvider

    provider = IQMProvider(base_url, token=token)
    backend = provider.get_backend()
    print(f"Connected to backend: {backend.name}")
    return backend

## Run comparison

1. Train or copy checkpoints into [`setup/cross_validation/Models/Weights/depth 2/`](setup/cross_validation/Models/Weights/depth%202/) (Odra `epoch_30`, Simulator **`epoch_30_ideal`** per fold).
2. Optional: set **`RUN_IQM_HARDWARE = True`**, **`EVAL_SHOTS`**, and **`N_REPEATS`** (≥10) for IQM (token prompt).
3. With **`RUN_IQM_HARDWARE = False`**, only **statevector** accuracy/F1 are computed for each fold and ansatz.
4. Results: **`tests/ansatz_odra_evaluation/outputs/run_<timestamp>/`** — `run_level_results.csv` is empty if IQM was skipped.



In [ ]:
if RUN_IQM_HARDWARE:
    if EVAL_SHOTS <= 0:
        raise ValueError("EVAL_SHOTS must be positive.")

_missing = []
for fold in range(1, K_FOLDS + 1):
    tp = fold_test_csv_path(fold)
    if not tp.is_file():
        _missing.append(str(tp))
    for wp in (odra_weights_path(fold), simulator_ideal_weights_path(fold)):
        if not wp.is_file():
            _missing.append(str(wp))
if _missing:
    raise FileNotFoundError(
        "Missing CV artifacts (run Training notebooks under setup/cross_validation/Models/Training/depth 2/ or place files manually): "
        + "; ".join(_missing)
    )

try:
    from IPython.display import display
except ImportError:
    display = print

run_stamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%d_%H%M%S")
out_dir = ROOT / "tests" / "ansatz_odra_evaluation" / "outputs" / f"run_{run_stamp}"
out_dir.mkdir(parents=True, exist_ok=True)

runs_csv = out_dir / "run_level_results.csv"
summary_csv = out_dir / "summary_comparison.csv"
_runs_csv_has_rows = False


def _checkpoint_summary() -> None:
    """Rewrite summary CSV after each fold × ansatz (all rows so far)."""
    pd.DataFrame(summary_rows).to_csv(summary_csv, index=False)


def _checkpoint_run_row() -> None:
    """Append latest IQM repeat row (survives crashes mid-repeat loop)."""
    global _runs_csv_has_rows
    pd.DataFrame([run_rows[-1]]).to_csv(
        runs_csv,
        mode="a" if _runs_csv_has_rows else "w",
        header=not _runs_csv_has_rows,
        index=False,
    )
    _runs_csv_has_rows = True


ansatz_specs = [
    ("odra", odra_ansatz, odra_weights_path),
    ("simulator", simulator_ansatz, simulator_ideal_weights_path),
]

run_rows = []
summary_rows = []

print(f"CV depth {ANSATZ_DEPTH}, folds 1..{K_FOLDS} → outputs: {out_dir}")
print(f"Checkpoints: IQM repeats → append {runs_csv.name}; each ansatz done → rewrite {summary_csv.name}")

iqm_backend = connect_to_iqm_backend() if RUN_IQM_HARDWARE else None

for fold in range(1, K_FOLDS + 1):
    test_csv = fold_test_csv_path(fold)
    X_test, y_test = load_fold_test_data(ROOT, CROSS_VAL_DIR, fold)
    X_tensor = torch.tensor(X_test, dtype=torch.float32)
    n_samples = len(y_test)

    for ansatz_name, ansatz_fn, weight_path_fn in ansatz_specs:
        weight_path = weight_path_fn(fold)
        print(f"\n--- fold {fold} | {ansatz_name} ---")

        sv_model = build_statevector_model(ansatz_fn, NUM_QUBITS, ANSATZ_DEPTH)
        load_checkpoint_hybrid(sv_model, weight_path)
        sv_model.eval()
        with torch.no_grad():
            sv_pred = sv_model(X_tensor).detach().cpu().numpy().flatten()
        sv_labels = predictions_to_labels(sv_pred)
        sv_acc = float(accuracy_score(y_test, sv_labels))
        sv_f1 = float(f1_score(y_test, sv_labels, pos_label=1))

        if iqm_backend is not None:
            iqm_accs = []
            iqm_f1s = []

            repeat_iter = range(N_REPEATS)
            if HAS_TQDM:
                repeat_iter = tqdm(repeat_iter, desc=f"IQM fold{fold} {ansatz_name}")

            for rep in repeat_iter:
                hw_model, hw_est = build_iqm_model(
                    iqm_backend, ansatz_fn, EVAL_SHOTS, NUM_QUBITS, ANSATZ_DEPTH
                )
                load_checkpoint_connector(hw_model, weight_path)
                hw_model.eval()
                t0 = time.time()
                with torch.no_grad():
                    hw_pred = hw_model(X_tensor).detach().cpu().numpy().flatten()
                _wall = time.time() - t0
                hw_labels = predictions_to_labels(hw_pred)
                acc = float(accuracy_score(y_test, hw_labels))
                f1 = float(f1_score(y_test, hw_labels, pos_label=1))
                iqm_accs.append(acc)
                iqm_f1s.append(f1)

                run_rows.append(
                    {
                        "timestamp_utc": pd.Timestamp.now(tz="UTC").isoformat(),
                        "fold": fold,
                        "ansatz": ansatz_name,
                        "shots": EVAL_SHOTS,
                        "repeat_index": rep,
                        "accuracy": acc,
                        "f1": f1,
                        "weight_path": str(weight_path),
                        "test_csv": str(test_csv),
                        "n_samples": n_samples,
                        "qpu_time_total": float(hw_est.total_qpu_time),
                        "wall_time_forward_s": float(_wall),
                    }
                )
                _checkpoint_run_row()

            mean_acc = float(np.mean(iqm_accs))
            mean_f1 = float(np.mean(iqm_f1s))
            std_acc = float(np.std(iqm_accs, ddof=1)) if N_REPEATS > 1 else 0.0
            std_f1 = float(np.std(iqm_f1s, ddof=1)) if N_REPEATS > 1 else 0.0
        else:
            mean_acc = mean_f1 = std_acc = std_f1 = float("nan")

        summary_rows.append(
            {
                "fold": fold,
                "ansatz": ansatz_name,
                "statevector_accuracy": sv_acc,
                "statevector_f1": sv_f1,
                "statevector_std_accuracy": 0.0,
                "statevector_std_f1": 0.0,
                "iqm_mean_accuracy": mean_acc,
                "iqm_std_accuracy": std_acc,
                "iqm_mean_f1": mean_f1,
                "iqm_std_f1": std_f1,
                "eval_shots": EVAL_SHOTS if RUN_IQM_HARDWARE else float("nan"),
                "n_repeats": N_REPEATS if RUN_IQM_HARDWARE else 0,
                "test_csv": str(test_csv),
                "weight_path": str(weight_path),
            }
        )
        _checkpoint_summary()

        print(f"Statevector  acc={sv_acc:.4f} f1={sv_f1:.4f}")
        if iqm_backend is not None:
            print(f"IQM mean±std acc={mean_acc:.4f}±{std_acc:.4f}  f1={mean_f1:.4f}±{std_f1:.4f}")
        else:
            print("IQM skipped (RUN_IQM_HARDWARE=False)")

df_runs = pd.DataFrame(run_rows)
df_summary = pd.DataFrame(summary_rows)

df_runs.to_csv(runs_csv, index=False)
df_summary.to_csv(summary_csv, index=False)

print(f"\nFinal save: {runs_csv}")
print(f"Final save: {summary_csv}")
display(df_summary)


In [ ]:
if "df_summary" not in globals():
    raise RuntimeError("Run the comparison cell above first (defines df_summary).")

# Mean ± std of statevector metrics across folds (per ansatz)
g = df_summary.groupby("ansatz", as_index=False)
agg = g.agg(
    statevector_accuracy_mean=("statevector_accuracy", "mean"),
    statevector_accuracy_std=("statevector_accuracy", "std"),
    statevector_f1_mean=("statevector_f1", "mean"),
    statevector_f1_std=("statevector_f1", "std"),
)
if RUN_IQM_HARDWARE and df_summary["iqm_mean_accuracy"].notna().any():
    agg2 = g.agg(
        iqm_mean_accuracy_mean=("iqm_mean_accuracy", "mean"),
        iqm_mean_accuracy_std=("iqm_mean_accuracy", "std"),
        iqm_mean_f1_mean=("iqm_mean_f1", "mean"),
        iqm_mean_f1_std=("iqm_mean_f1", "std"),
    )
    agg = agg.merge(agg2, on="ansatz")
display(agg)
